In [2]:
import re
import os
import shutil

In [3]:
def get_best_components(log_file_path):
    # 定义正则表达式，用于匹配所需内容
    test_dataset_pattern = re.compile(r'test dataset:\s*(.*)')
    pred_len_1_pattern = re.compile(r'pred_len_1:\s*(.*)')
    pred_len_2_pattern = re.compile(r'pred_len_2:\s*(.*)')
    best_components_pattern = re.compile(r'best components:\s*(.*)')

    # 用于存储所有匹配的结果
    test_datasets = []
    best_components = []

    with open(log_file_path, 'r') as file:
        # 读取整个文件
        log_content = file.read()

        # 查找所有匹配的内容
        test_datasets = test_dataset_pattern.findall(log_content)
        pred_len_1 = pred_len_1_pattern.findall(log_content)
        pred_len_2 = pred_len_2_pattern.findall(log_content)
        best_components = best_components_pattern.findall(log_content)

    pred_len = []
    # 打印所有匹配的内容
    if test_datasets:
        print("Test datasets:")
        for i, dataset in enumerate(test_datasets):
            print(dataset)
            if dataset != 'ili':
                print(int(pred_len_1[i]))
                pred_len.append(int(pred_len_1[i]))
            else:
                print(int(pred_len_2[i]))
                pred_len.append(int(pred_len_2[i]))
    else:
        print("No 'test dataset:' found in the log.")

    if best_components:
        print("Best components:")
        for component in best_components:
            print(component)
    else:
        print("No 'best components:' found in the log.")\
    
    return test_datasets, pred_len, best_components

# 分布漂移 最优TSGym配置SH生成

In [ ]:
# 读取日志文件并提取所有匹配的内容
log_file_path = '../meta_roll.log'  # 请根据实际情况修改日志文件路径
test_datasets, best_components = get_best_components(log_file_path)

# 模板文件路径
template_path = f'/data/nishome/user1/minqi/TSGym/scripts/long_term_forecast/ETTh2_script/TSGym_ETTh2.sh'
    
# 输出目录
output_dir = f'/data/nishome/user1/minqi/TSGym/scripts/long_term_forecast_rolling/ETTh2_scripts_roll'
script_contents = []
for i, dataset in enumerate(test_datasets):

    # 确保输出目录存在
    # if os.path.exists(output_dir):
    #     print('delete current folder!')
    #     shutil.rmtree(output_dir)
    os.makedirs(output_dir, exist_ok=True)

    # 读取模板内容
    with open(template_path, 'r') as file:
        template_content = file.read()

    # 对于每个模型名称，生成一个 shell 脚本
    setting = '_'.join(best_components[i].split('_')[2:])
    file_name = setting
    
    model_name = '_'.join(setting.split('_')[:12])

    intros = ['x_mark', 'multi-granularity', 'Normalization', 'Decomposition', 'Channel-independent',
            'Tokenization', 'Backbone', 'Attention', 'Feature-Attention', 'Encoder-only', 'Frozen', 'Dataset',
            'Sequence Length', 'd_model', 'd_ff', 'Encoder layers',
            'Training Epochs', 'Loss Function', 'Learning Rate', 'Learning Rate Strategy']
    key = '_'.join([setting[setting.find('TSGym')+6: setting.find('ftM')-1],
                                            re.search(r'sl(\d+)', setting)[1],
                                            re.search(r'dm(\d+)', setting)[1],
                                            re.search(r'df(\d+)', setting)[1],
                                            re.search(r'el(\d+)', setting)[1],
                                            re.search(r'epochs(\d+)', setting)[1],
                                            re.search(r'lf([A-Za-z]+)', setting)[1],
                                            re.search(r'lr(\d+(?:\.\d+)?)', setting)[1],
                                            re.search(r'lrs([A-Za-z]+\d*)', setting)[1]])
    record = {}
    for n, param in enumerate(key.split('_')):
        record.update({intros[n]:param})

    # 替换模型名称
    script_content = template_content.replace('$model_name', model_name)
    script_content = script_content.replace(f'$seq_len', record['Sequence Length'])
    script_content = script_content.replace(f'$d_model', record['d_model'])
    script_content = script_content.replace(f'$d_ff', record['d_ff'])
    script_content = script_content.replace(f'$e_layers', record['Encoder layers'])
    script_content = script_content.replace(f'$train_epochs', record['Training Epochs'])
    script_content = script_content.replace(f'$loss', record['Loss Function'])
    script_content = script_content.replace(f'$learning_rate', record['Learning Rate'])
    script_content = script_content.replace(f'$lradj', record['Learning Rate Strategy'])
    _, train_start, val_start, test_end = re.split(r'[-+]', dataset)
    script_content = script_content.replace(f'$train_start', train_start)
    script_content = script_content.replace(f'$val_start', val_start)
    script_content = script_content.replace(f'$test_end', test_end)
    script_contents.append(script_content)
    
# 定义输出文件名
output_file = os.path.join(output_dir, f'TSGym_roll.sh')

# 写入新的 shell 脚本
with open(output_file, 'w') as file:
    # 写入 shell 脚本的 shebang
    file.write("#!/bin/bash\n\n")
    
    # 写入每个 Python 命令
    for command in script_contents:
        file.write(command + "\n\n")

# print(f'Generated {output_file}')

# AFAC 最优TSGym配置SH生成

In [2]:
test_datasets = ['afac+pad20250718']
best_components = ['Exchange_LTF_TSGym_True_False_Stat_None_False_inverted-encoding_MLP_null_self-attention_True_False_custom_ftM_sl96_ll48_pl96_dm256_el3_dl1_df1024_fc3_ebtimeF_dtTrue_Exp_epochs50_lfMAE_lr0.001_lrsnull_0']

In [4]:
# 模板文件路径
template_path = f'/data/nishome/user1/minqi/TSGym/scripts/long_term_forecast/afac_script/TSGym.sh'
    
# 输出目录
output_dir = f'/data/nishome/user1/minqi/TSGym/scripts/long_term_forecast/afac_script'
script_contents = []
for i, dataset in enumerate(test_datasets):

    # 确保输出目录存在
    # if os.path.exists(output_dir):
    #     print('delete current folder!')
    #     shutil.rmtree(output_dir)
    os.makedirs(output_dir, exist_ok=True)

    # 读取模板内容
    with open(template_path, 'r') as file:
        template_content = file.read()

    # 对于每个模型名称，生成一个 shell 脚本
    setting = '_'.join(best_components[i].split('_')[2:])
    file_name = setting
    
    model_name = '_'.join(setting.split('_')[:12])

    intros = ['x_mark', 'multi-granularity', 'Normalization', 'Decomposition', 'Channel-independent',
            'Tokenization', 'Backbone', 'Attention', 'Feature-Attention', 'Encoder-only', 'Frozen', 'Dataset',
            'Sequence Length', 'd_model', 'd_ff', 'Encoder layers',
            'Training Epochs', 'Loss Function', 'Learning Rate', 'Learning Rate Strategy']
    key = '_'.join([setting[setting.find('TSGym')+6: setting.find('ftM')-1],
                                            re.search(r'sl(\d+)', setting)[1],
                                            re.search(r'dm(\d+)', setting)[1],
                                            re.search(r'df(\d+)', setting)[1],
                                            re.search(r'el(\d+)', setting)[1],
                                            re.search(r'epochs(\d+)', setting)[1],
                                            re.search(r'lf([A-Za-z]+)', setting)[1],
                                            re.search(r'lr(\d+(?:\.\d+)?)', setting)[1],
                                            re.search(r'lrs([A-Za-z]+\d*)', setting)[1]])
    record = {}
    for n, param in enumerate(key.split('_')):
        record.update({intros[n]:param})

    # 替换模型名称
    script_content = template_content.replace('$model_name', model_name)
    script_content = script_content.replace(f'$seq_len', record['Sequence Length'])
    script_content = script_content.replace(f'$d_model', record['d_model'])
    script_content = script_content.replace(f'$d_ff', record['d_ff'])
    script_content = script_content.replace(f'$e_layers', record['Encoder layers'])
    script_content = script_content.replace(f'$train_epochs', record['Training Epochs'])
    script_content = script_content.replace(f'$loss', record['Loss Function'])
    script_content = script_content.replace(f'$learning_rate', record['Learning Rate'])
    script_content = script_content.replace(f'$lradj', record['Learning Rate Strategy'])
    script_content = script_content.replace('$dataset', dataset)
    script_content = script_content.replace('$model_id', f'{dataset}_14_7')
    script_contents.append(script_content)
    
# 定义输出文件名
output_file = os.path.join(output_dir, f'TSGym_best.sh')

# 写入新的 shell 脚本
with open(output_file, 'w') as file:
    # 写入 shell 脚本的 shebang
    file.write("#!/bin/bash\n\n")
    
    # 写入每个 Python 命令
    for command in script_contents:
        file.write(command + "\n\n")

# print(f'Generated {output_file}')

# Optuna 最优TSGym配置SH生成

In [9]:
# 读取日志文件并提取所有匹配的内容
log_file_path = '/data/nishome/user1/minqi/TSGym/logfiles/meta_4random.log'  # 请根据实际情况修改日志文件路径
test_datasets, pred_lens, best_components = get_best_components(log_file_path)

Test datasets:
Exchange
96
Exchange
192
Exchange
336
Exchange
720
ili
24
ili
36
ili
48
ili
60
ECL
96
ECL
192
ECL
336
ECL
720
weather
96
weather
192
weather
336
weather
720
Best components:
ili_LTF_TSGym_True_False_Stat_None_False_inverted-encoding_MLP_null_self-attention_True_False_custom_ftM_sl512_ll48_pl24_dm64_el3_dl1_df256_fc3_ebtimeF_dtTrue_Exp_epochs10_lfMAE_lr0.001_lrsnull_0
ili_LTF_TSGym_False_False_RevIN_DFT_False_inverted-encoding_MLP_null_sparse-attention_True_False_custom_ftM_sl192_ll48_pl36_dm256_el2_dl1_df1024_fc3_ebtimeF_dtTrue_Exp_epochs10_lfHUBER_lr0.001_lrstype1_0
ili_LTF_TSGym_True_False_RevIN_MoEMA_False_inverted-encoding_GRU_null_frequency-enhanced-attention_True_False_custom_ftM_sl192_ll48_pl48_dm256_el2_dl1_df1024_fc3_ebtimeF_dtTrue_Exp_epochs10_lfHUBER_lr0.001_lrstype1_0
ili_LTF_TSGym_False_False_RevIN_None_True_series-patching_MLP_null_null_True_False_custom_ftM_sl512_ll48_pl60_dm64_el2_dl1_df256_fc3_ebtimeF_dtTrue_Exp_epochs50_lfMSE_lr0.001_lrstype1_0
Exchange

In [10]:
def gen_best_sh(test_datasets, pred_lens, best_components, output_dir_name='exp_random_meta', delete=False, devices='0'):
    # script_contents = []
    for i, test_dataset in enumerate(test_datasets):
        dataset_foldor = test_dataset.replace('ili', 'ILI').replace('weather', 'Weather')
        template_path = f'/data/nishome/user1/minqi/TSGym/scripts/long_term_forecast/{dataset_foldor}_script/TSGym.sh'
        output_dir = f'/data/nishome/user1/minqi/TSGym/scripts/{output_dir_name}'
        if delete:
            # 确保输出目录存在
            if os.path.exists(output_dir):
                print('delete current folder!')
                shutil.rmtree(output_dir)
        os.makedirs(output_dir, exist_ok=True)

        # 读取模板内容
        with open(template_path, 'r') as file:
            template_content = file.read()

        # 对于每个模型名称，生成一个 shell 脚本
        setting = '_'.join(best_components[i].split('_')[2:])
        file_name = setting
        
        model_name = '_'.join(setting.split('_')[:12])

        intros = ['x_mark', 'multi-granularity', 'Normalization', 'Decomposition', 'Channel-independent',
                'Tokenization', 'Backbone', 'Attention', 'Feature-Attention', 'Encoder-only', 'Frozen', 'Dataset',
                'Sequence Length', 'd_model', 'd_ff', 'Encoder layers',
                'Training Epochs', 'Loss Function', 'Learning Rate', 'Learning Rate Strategy']
        key = '_'.join([setting[setting.find('TSGym')+6: setting.find('ftM')-1],
                                                re.search(r'sl(\d+)', setting)[1],
                                                re.search(r'dm(\d+)', setting)[1],
                                                re.search(r'df(\d+)', setting)[1],
                                                re.search(r'el(\d+)', setting)[1],
                                                re.search(r'epochs(\d+)', setting)[1],
                                                re.search(r'lf([A-Za-z]+)', setting)[1],
                                                re.search(r'lr(\d+(?:\.\d+)?)', setting)[1],
                                                re.search(r'lrs([A-Za-z]+\d*)', setting)[1]])
        record = {}
        for n, param in enumerate(key.split('_')):
            record.update({intros[n]:param})

        # 替换模型名称
        script_content = template_content.replace('$model_name', model_name)
        script_content = script_content.replace(f'$seq_len', record['Sequence Length'])
        script_content = script_content.replace(f'$d_model', record['d_model'])
        script_content = script_content.replace(f'$d_ff', record['d_ff'])
        script_content = script_content.replace(f'$e_layers', record['Encoder layers'])
        script_content = script_content.replace(f'$train_epochs', record['Training Epochs'])
        script_content = script_content.replace(f'$loss', record['Loss Function'])
        script_content = script_content.replace(f'$learning_rate', record['Learning Rate'])
        script_content = script_content.replace(f'$lradj', record['Learning Rate Strategy'])
        # script_contents.append(script_content)
        
        script_contents = script_content.split('\n\n')
        for _ in script_contents:
            if f'--pred_len {pred_lens[i]}' in _:
                script_content = f'CUDA_VISIBLE_DEVICES={devices} '+ _
                break
        # 定义输出文件名
        output_file = os.path.join(output_dir, f'{test_dataset}_{pred_lens[i]}.sh')

        # 写入新的 shell 脚本
        with open(output_file, 'w') as file:
                file.write(script_content)

        print(f'test_dataset {test_dataset} pred_lens:{pred_lens[i]}')
        print(f'Generated {output_file}')


In [11]:
output_dir_name='exp_random_meta'
# gen_best_sh(test_datasets, pred_lens, best_components, output_dir_name=output_dir_name, delete=True, devices='1')
gen_best_sh(test_datasets, pred_lens, best_components, output_dir_name=output_dir_name, delete=False, devices='1')

test_dataset Exchange pred_lens:96
Generated /data/nishome/user1/minqi/TSGym/scripts/exp_random_meta/Exchange_96.sh
test_dataset Exchange pred_lens:192
Generated /data/nishome/user1/minqi/TSGym/scripts/exp_random_meta/Exchange_192.sh
test_dataset Exchange pred_lens:336
Generated /data/nishome/user1/minqi/TSGym/scripts/exp_random_meta/Exchange_336.sh
test_dataset Exchange pred_lens:720
Generated /data/nishome/user1/minqi/TSGym/scripts/exp_random_meta/Exchange_720.sh
test_dataset ili pred_lens:24
Generated /data/nishome/user1/minqi/TSGym/scripts/exp_random_meta/ili_24.sh
test_dataset ili pred_lens:36
Generated /data/nishome/user1/minqi/TSGym/scripts/exp_random_meta/ili_36.sh
test_dataset ili pred_lens:48
Generated /data/nishome/user1/minqi/TSGym/scripts/exp_random_meta/ili_48.sh
test_dataset ili pred_lens:60
Generated /data/nishome/user1/minqi/TSGym/scripts/exp_random_meta/ili_60.sh
test_dataset ECL pred_lens:96
Generated /data/nishome/user1/minqi/TSGym/scripts/exp_random_meta/ECL_96.sh


In [12]:
# 读取日志文件并提取所有匹配的内容
log_file_path = '/data/nishome/user1/minqi/TSGym/logfiles/meta_4optuna.log'  # 请根据实际情况修改日志文件路径
test_datasets, pred_lens, best_components = get_best_components(log_file_path)

Test datasets:
Exchange
96
Exchange
192
Exchange
336
Exchange
720
ili
24
ili
36
ili
48
ili
60
ECL
96
ECL
192
ECL
336
ECL
720
weather
96
weather
192
weather
336
weather
720
Best components:
weather_LTF_TSGym_True_True_RevIN_None_True_series-encoding_MLP_null_null_True_False_custom_ftM_sl512_ll48_pl96_dm256_el3_dl1_df1024_fc3_ebtimeF_dtTrue_Exp_epochs10_lfMAE_lr0.001_lrsnull_0
ECL_LTF_TSGym_False_False_RevIN_None_False_inverted-encoding_MLP_null_frequency-enhanced-attention_True_False_custom_ftM_sl512_ll48_pl192_dm256_el2_dl1_df1024_fc3_ebtimeF_dtTrue_Exp_epochs10_lfMAE_lr0.001_lrstype1_0
weather_LTF_TSGym_False_False_RevIN_MA_True_series-patching_MLP_null_null_True_False_custom_ftM_sl512_ll48_pl336_dm256_el3_dl1_df1024_fc3_ebtimeF_dtTrue_Exp_epochs50_lfMAE_lr0.001_lrstype1_0
weather_LTF_TSGym_False_False_DishTS_DFT_True_series-patching_GRU_null_null_True_False_custom_ftM_sl512_ll48_pl720_dm256_el3_dl1_df1024_fc3_ebtimeF_dtTrue_Exp_epochs50_lfMAE_lr0.001_lrsnull_0
Exchange_LTF_TSGym_Fals

In [13]:
output_dir_name='exp_optuna_meta'
# gen_best_sh(test_datasets, pred_lens, best_components, output_dir_name=output_dir_name, delete=True, devices='0')
gen_best_sh(test_datasets, pred_lens, best_components, output_dir_name=output_dir_name, delete=False, devices='0')

test_dataset Exchange pred_lens:96
Generated /data/nishome/user1/minqi/TSGym/scripts/exp_optuna_meta/Exchange_96.sh
test_dataset Exchange pred_lens:192
Generated /data/nishome/user1/minqi/TSGym/scripts/exp_optuna_meta/Exchange_192.sh
test_dataset Exchange pred_lens:336
Generated /data/nishome/user1/minqi/TSGym/scripts/exp_optuna_meta/Exchange_336.sh
test_dataset Exchange pred_lens:720
Generated /data/nishome/user1/minqi/TSGym/scripts/exp_optuna_meta/Exchange_720.sh
test_dataset ili pred_lens:24
Generated /data/nishome/user1/minqi/TSGym/scripts/exp_optuna_meta/ili_24.sh
test_dataset ili pred_lens:36
Generated /data/nishome/user1/minqi/TSGym/scripts/exp_optuna_meta/ili_36.sh
test_dataset ili pred_lens:48
Generated /data/nishome/user1/minqi/TSGym/scripts/exp_optuna_meta/ili_48.sh
test_dataset ili pred_lens:60
Generated /data/nishome/user1/minqi/TSGym/scripts/exp_optuna_meta/ili_60.sh
test_dataset ECL pred_lens:96
Generated /data/nishome/user1/minqi/TSGym/scripts/exp_optuna_meta/ECL_96.sh


# 解析meta-predictor选出来的模型

In [ ]:
def get_meta_model_path(log_file_path):
    # 定义正则表达式，用于匹配所需内容
    test_dataset_pattern = re.compile(r'test dataset:\s*(.*)')
    pred_len_1_pattern = re.compile(r'pred_len_1:\s*(.*)')
    pred_len_2_pattern = re.compile(r'pred_len_2:\s*(.*)')
    best_components_pattern = re.compile(r'best components:\s*(.*)')

    # 用于存储所有匹配的结果
    test_datasets = []
    best_components = []

    with open(log_file_path, 'r') as file:
        # 读取整个文件
        log_content = file.read()

        # 查找所有匹配的内容
        test_datasets = test_dataset_pattern.findall(log_content)
        pred_len_1 = pred_len_1_pattern.findall(log_content)
        pred_len_2 = pred_len_2_pattern.findall(log_content)
        best_components = best_components_pattern.findall(log_content)

    pred_len = []
    # 打印所有匹配的内容
    if test_datasets:
        print("Test datasets:")
        for i, dataset in enumerate(test_datasets):
            print(dataset)
            if dataset != 'ili':
                print(int(pred_len_1[i]))
                pred_len.append(int(pred_len_1[i]))
            else:
                print(int(pred_len_2[i]))
                pred_len.append(int(pred_len_2[i]))
    else:
        print("No 'test dataset:' found in the log.")

    if best_components:
        print("Best components:")
        for component in best_components:
            print(component)
    else:
        print("No 'best components:' found in the log.")\
    
    return test_datasets, pred_len, best_components

In [ ]:
# 读取日志文件并提取所有匹配的内容
log_file_path = '/data/nishome/user1/minqi/TSGym/tsgym_random_meta.log.log'  # 请根据实际情况修改日志文件路径
test_datasets, pred_lens, best_components = get_best_components(log_file_path)

In [ ]:
log_file_path = '/data/nishome/user1/minqi/TSGym/tsgym_optuna_meta.log'